In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
 
# Minimum corpus frequency to consider a token (filters hapax legomena)
MIN_FREQ = 5
 
# PMI threshold: tokens with avg PMI BELOW this are filler candidates.
# Fillers distribute independently of their neighbors → low PMI.
# Content words co-occur predictably → higher PMI.
# Calibrated on Indonesian FnB corpus; tune if your domain shifts.
PMI_THRESHOLD = 2.5
 
# Position entropy threshold: tokens with entropy ABOVE this appear at
# any sentence position (filler-like). Content words tend to cluster
# at subject/object/verb positions → lower entropy.
ENTROPY_THRESHOLD = 1.2
 
# Partial word regex — language-agnostic; matches any alphabetic prefix
# followed by a hyphen (na-, bi-, wou-). Never hardcoded vocabulary.
PARTIAL_WORD_PATTERN = r"^[a-zA-Z]+-$"

In [2]:
import re

# ─────────────────────────────────────────────────────────────────────────────
# LAYER 1: MORPHOLOGICAL POS TAGGER
# ─────────────────────────────────────────────────────────────────────────────
# All rules are based on Indonesian morphological structure and phonological
# shape — NOT on a fixed vocabulary list. Each rule generalises to any token
# that matches the morphological pattern, including tokens never seen in
# training.
#
# Rule priority (top → bottom, first match wins):
#   1. -nya suffix         → NOUN_VERB  (possessive nominaliser, always nominal)
#   2. Verb prefixes       → VERB       (me-, ber-, ter-, di-, mem-, men-, …)
#   3. -kan suffix         → VERB       (imperative/causative)
#   4. -an suffix          → NOUN_VERB  (nominal derivation)
#   5. Number words        → NUM
#   6. Pronouns            → PRON       (closed class)
#   7. Address terms       → ADDR       (vocative particles)
#   8. Interjection shape  → INTJ       (phonological rules A–D)
#   9. Particles           → PART       (closed class, kept small)
#  10. Repair/DM markers   → DM
#  11. Partial word        → PARTIAL    (hyphen-ending)
#  12. Default             → CONTENT
# ─────────────────────────────────────────────────────────────────────────────

_PRON = frozenset({
    "saya", "aku", "gue", "gw", "kami", "kita",
    "kamu", "anda", "dia", "mereka", "lo", "lu"
})
 
_ADDR = frozenset({
    "bang", "mas", "kak", "pak", "bu", "mbak", "bro", "sis",
})
 
_PART = frozenset({
    # Sentence-final and discourse particles
    "ya", "deh", "nih", "dong", "lah", "sih", "tuh",
    "aja", "gitu", "yaudah", "yuk", "kok", "pun", "lho", "nah", "ah",
})
 
_REPAIR = frozenset({
    # Repair / discourse markers that signal a correction is coming
    "bukan", "maaf", "maksudnya", "wait", "soalnya",
    "sebenarnya", "sebenernya", "intinya", "pokoknya",
})
 
_NUM_WORDS = frozenset({
    "satu", "dua", "tiga", "empat", "lima", "enam", "tujuh",
    "delapan", "sembilan", "sepuluh", "sebelas",
})
 
# Pre-compiled regex patterns for speed
_RE_NYA        = re.compile(r".+nya$")
_RE_VERB_PFX   = re.compile(r"^(me|ber|ter|di|mem|men|meng|meny).{2,}")
_RE_KAN        = re.compile(r".+kan$")
_RE_AN         = re.compile(r".+an$")
_RE_NUM        = re.compile(r"^\d+$")
_RE_INTJ_VOWEL = re.compile(r"^[aeiou][aeiou]*h?$")          # Rule A
_RE_INTJ_STEM  = re.compile(                                   # Rule B
    r"^(uh+|um+|er+|hmm+|hm+|mm+|eh+|ah+|oh+|ih+|ehm+|em+)$"
)
_RE_INTJ_RPT   = re.compile(r"^([aeiou])\1+$")               # Rule C
_RE_INTJ_KNOWN = re.compile(                                   # Rule D
    r"^(anu|aduh|waduh|astaga|duh|ups|oops|heh)$"
)
_RE_PARTIAL    = re.compile(PARTIAL_WORD_PATTERN)

In [3]:
def infer_pos(token: str) -> str:
    """
    Infer Indonesian POS tag from morphological and phonological rules.

    Returns one of:
        NOUN_VERB  — nominal (includes -nya / -an derivations)
        VERB       — verbal (morphological prefix or -kan imperative)
        NUM        — numeral
        PRON       — pronoun
        ADDR       — address/vocative term
        INTJ       — interjection / filled pause (by phonological shape)
        PART       — discourse/sentence particle (closed class)
        DM         — repair or discourse marker
        PARTIAL    — truncated word (ends with hyphen)
        CONTENT    — default: likely content word (NOUN, ADJ, ADV, …)
    """
    t = token.lower()

    # prioroty 1: close-class exact matches
    if t in _REPAIR:
        return "DM"
    if t in _PRON:
        return "PRON"
    if t in _ADDR:
        return "ADDR"
    if t in _PART:
        return "PART"
    if t in _NUM_WORDS or _RE_NUM.match(t):
        return "NUM"

    # Priority 2: Interjection by phonological shape (no vocabulary needed)
    if len(t) >= 2 and _RE_INTJ_VOWEL.match(t): 
        return "INTJ"  # A: pure vowel
    if _RE_INTJ_STEM.match(t):                    
        return "INTJ"  # B: stem+lengthening
    if _RE_INTJ_RPT.match(t):                     
        return "INTJ"  # C: vowel repeat
    if _RE_INTJ_KNOWN.match(t):                   
        return "INTJ"  # D: known stems

    # Priority 3: Partial word (hyphen-ending)
    if _RE_PARTIAL.match(token):        
        return "PARTIAL"

    # Priority 4: -nya suffix (always nominalises, before any prefix check)
    if len(t) > 4 and _RE_NYA.match(t): 
        return "NOUN_VERB"

    # Priority 5: Unambiguous verb prefixes (me/ber/ter/di/mem/men/meng/meny)
    #             pe- and se- excluded — also used as nominal prefixes
    if _RE_VERB_PFX.match(t):           
        return "VERB"

    # Priority 6: -kan suffix (causative/imperative — always verb)
    if len(t) > 5 and _RE_KAN.match(t): 
        return "VERB"

    # Priority 7: -an suffix WITHOUT a verb prefix (nominal derivation)
    #             Only reaches here if steps 5–6 did not match
    if len(t) > 4 and _RE_AN.match(t): 
        return "NOUN_VERB"

    return "CONTENT"

In [4]:
import json

raw_records = []
input_path = "../data/synthetic_id_formal_informal_normalized.jsonl"
with open(input_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_records.append(json.loads(line))
raw_records[0]

{'id': 1,
 'text': 'Tolong pesankan nasi kuning tiga porsi, untuk dibawa pulang ya.',
 'style': 'formal',
 'domain': 'restoran',
 'text_normalized': 'tolong pesankan nasi kuning tiga porsi untuk dibawa pulang ya'}

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# LAYER 2: CORPUS STATISTICS (PMI + POSITION ENTROPY)
# ─────────────────────────────────────────────────────────────────────────────

def tokenize(text: str) -> list[str]:
    text = text.lower().strip()
    text = re.sub(r"[,\.!\?]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.split()

test_text = raw_records[0]["text_normalized"]
tokenize(test_text)

['tolong',
 'pesankan',
 'nasi',
 'kuning',
 'tiga',
 'porsi',
 'untuk',
 'dibawa',
 'pulang',
 'ya']

In [6]:
import math
from collections import Counter, defaultdict

def compute_corpus_stats(sentences: list[list[str]], min_freq: int = MIN_FREQ) -> dict[str, dict]:
    """
    Compute two statistics for every token with freq >= min_freq:

    avg_pmi:
        Average pointwise mutual information with neighboring tokens.
        PMI(w1,w2) = log2[ P(w1,w2) / (P(w1) * P(w2)) ]
        Low avg_pmi → token distributes independently of context → filler-like.

    pos_entropy:
        Shannon entropy of the token's normalized position distribution
        across sentences (positions bucketed into 5 equal-width bins).
        High entropy → token appears at any position → filler-like.
        Low entropy → token clusters at specific positions → content-like.

    Returns dict: token → {"freq": int, "avg_pmi": float, "pos_entropy": float}
    """
    token_freq = Counter(t for s in sentences for t in s)
    bigram_freq   = Counter()
    total_tokens = sum(len(s) for s in sentences)
    total_bigrams = 0

    pos_counts = defaultdict(Counter)   # token → {bucket: count}

    for sent in sentences:
        n = max(len(sent), 1)
        for i, tok in enumerate(sent):
            bucket = min(4, int(i / n * 5))    # 5 position buckets: 0–4
            pos_counts[tok][bucket] += 1

        for i in range(len(sent) - 1):
            bigram_freq[(sent[i], sent[i + 1])] += 1
            total_bigrams += 1

    # Compute avg PMI per token
    token_pmi_accum = defaultdict(list)
    for (w1, w2), cnt in bigram_freq.items():
        p_bigram = cnt / total_bigrams
        p_w1     = token_freq[w1] / total_tokens
        p_w2     = token_freq[w2] / total_tokens
        if p_w1 > 0 and p_w2 > 0 and p_bigram > 0:
            pmi = math.log2(p_bigram / (p_w1 * p_w2))
            token_pmi_accum[w1].append(pmi)
            token_pmi_accum[w2].append(pmi)

    avg_pmi = {
        tok: sum(pmis) / len(pmis)
        for tok, pmis in token_pmi_accum.items()
    }

    # Compute position entropy per token
    def entropy(counter: Counter) -> float:
        total = sum(counter.values())
        if total == 0:
            return 0.0
        return -sum(
            (c / total) * math.log2(c / total)
            for c in counter.values() if c > 0
        )

    pos_entropy = {tok: entropy(pos_counts[tok]) for tok in token_freq}

    # Assemble result
    stats = {}
    for tok, freq in token_freq.items():
        if freq < min_freq:
            continue
        stats[tok] = {
            "freq"       : freq,
            "avg_pmi"    : avg_pmi.get(tok, 0.0),
            "pos_entropy": pos_entropy.get(tok, 0.0),
        }

    return stats

In [7]:
def is_statistical_filler(
    stats          : dict,
    pmi_threshold  : float = PMI_THRESHOLD,
    entropy_threshold: float = ENTROPY_THRESHOLD,
) -> bool:
    """
    Returns True if a token's corpus statistics match the filler signature:
        avg_pmi    < pmi_threshold       (distributes independently)
    OR
        pos_entropy > entropy_threshold  (appears at any position)

    Using OR (not AND) increases recall — we'd rather flag a content word
    as a filler candidate and let Layer 1 POS arbitrate than miss real fillers.
    """
    return (
        stats["avg_pmi"] < pmi_threshold
        or stats["pos_entropy"] > entropy_threshold
    )

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# VOCABULARY BUILDER
# ─────────────────────────────────────────────────────────────────────────────

def build_vocabulary(
    sentences        : list[list[str]],
    min_freq         : int   = MIN_FREQ,
    pmi_threshold    : float = PMI_THRESHOLD,
    entropy_threshold: float = ENTROPY_THRESHOLD,
) -> dict:
    """
    Build the dynamic vocabulary by combining Layer 1 (POS) and Layer 2 (PMI).

    Classification logic per token:
        INTJ (POS)  → filled_pause          (high confidence)
        DM   (POS)  → repair_bridge         (high confidence)
        PART (POS)  → discourse_marker      (high confidence)
        CONTENT (POS) + statistical filler  → discourse_marker (low confidence)
        CONTENT (POS) + not filler          → excluded

    PRON, VERB, NUM, ADDR, NOUN_VERB, PARTIAL, CONTENT → not fillers
    (except CONTENT with statistical filler signal, see above)

    Returns a dict ready to be serialized as vocabulary.json.
    """
    corpus_stats = compute_corpus_stats(sentences, min_freq)

    filled_pauses     = []
    repair_bridges    = []
    discourse_markers = []

    for tok, stats in corpus_stats.items():
        pos = infer_pos(tok)

        if pos == "INTJ":
            filled_pauses.append(tok)

        elif pos == "DM":
            repair_bridges.append(tok)

        elif pos == "PART":
            discourse_markers.append(tok)

        # CONTENT words are NOT reclassified as discourse markers.
        # Food items, action verbs, and common content words share the same
        # statistical fingerprint (high entropy, moderate PMI) as discourse
        # markers in a domain corpus — the statistical signal cannot
        # distinguish them reliably. Discourse markers come only from the
        # PART and DM closed classes identified by morphological rules.

        # All other POS classes are content words — excluded from filler vocab

    # Sort all lists: repair bridges longest-first (for greedy phrase matching)
    filled_pauses.sort()
    repair_bridges     = sorted(repair_bridges, key=len, reverse=True)
    discourse_markers  = sorted(discourse_markers, key=len, reverse=True)

    return {
        "filled_pauses"    : filled_pauses,
        "repair_bridges"   : repair_bridges,
        "discourse_markers": discourse_markers,
        "partial_word_re"  : PARTIAL_WORD_PATTERN,
        "metadata": {
            "n_corpus_sentences": len(sentences),
            "n_tokens_analyzed" : len(corpus_stats),
            "pmi_threshold"     : pmi_threshold,
            "entropy_threshold" : entropy_threshold,
            "min_freq"          : min_freq,
        },
    }

# ─────────────────────────────────────────────────────────────────────────────
# REPORTING
# ─────────────────────────────────────────────────────────────────────────────

def print_report(vocab: dict, corpus_stats: dict) -> None:
    """Print a human-readable summary of the built vocabulary."""

    print("\n" + "=" * 70)
    print("VOCABULARY REPORT")
    print("=" * 70)
    meta = vocab["metadata"]
    print(f"  Corpus sentences  : {meta['n_corpus_sentences']}")
    print(f"  Tokens analyzed   : {meta['n_tokens_analyzed']}")
    print(f"  PMI threshold     : {meta['pmi_threshold']}")
    print(f"  Entropy threshold : {meta['entropy_threshold']}")
    print(f"  Min frequency     : {meta['min_freq']}")

    print(f"\n  Filled pauses ({len(vocab['filled_pauses'])} tokens):")
    for tok in vocab["filled_pauses"]:
        s = corpus_stats.get(tok, {})
        print(f"    {tok:20s}  freq={s.get('freq','?'):>5}  "
              f"pmi={s.get('avg_pmi',0):.3f}  "
              f"ent={s.get('pos_entropy',0):.3f}  "
              f"pos={infer_pos(tok)}")

    print(f"\n  Repair bridges ({len(vocab['repair_bridges'])} tokens):")
    for tok in vocab["repair_bridges"]:
        s = corpus_stats.get(tok, {})
        print(f"    {tok:20s}  freq={s.get('freq','?'):>5}  "
              f"pmi={s.get('avg_pmi',0):.3f}  "
              f"pos={infer_pos(tok)}")

    print(f"\n  Discourse markers ({len(vocab['discourse_markers'])} tokens):")
    for tok in vocab["discourse_markers"]:
        s = corpus_stats.get(tok, {})
        print(f"    {tok:20s}  freq={s.get('freq','?'):>5}  "
              f"pmi={s.get('avg_pmi',0):.3f}  "
              f"ent={s.get('pos_entropy',0):.3f}  "
              f"pos={infer_pos(tok)}")

    print(f"\n  Partial word regex: {vocab['partial_word_re']}")


def print_pos_breakdown(sentences: list[list[str]], min_freq: int) -> None:
    """Print POS tag distribution across the corpus."""
    token_freq = Counter(t for s in sentences for t in s)
    pos_token_counts = defaultdict(list)

    for tok, freq in token_freq.items():
        if freq < min_freq:
            continue
        pos = infer_pos(tok)
        pos_token_counts[pos].append((tok, freq))

    print("\n" + "=" * 70)
    print("POS TAG DISTRIBUTION (corpus tokens, freq >= min_freq)")
    print("=" * 70)

    pos_order = ["INTJ", "PART", "DM", "PARTIAL",
                 "PRON", "ADDR", "VERB", "NOUN_VERB", "NUM", "CONTENT"]

    for pos in pos_order:
        entries = sorted(pos_token_counts.get(pos, []), key=lambda x: -x[1])
        total_freq = sum(f for _, f in entries)
        print(f"\n  {pos} ({len(entries)} types, {total_freq} tokens):")
        for tok, freq in entries[:15]:   # show top 15 per class
            print(f"    {tok:20s}  freq={freq}")
        if len(entries) > 15:
            print(f"    … and {len(entries) - 15} more")

In [9]:
def run_pos_tests() -> bool:
    """
    Regression tests for infer_pos().
    Tests cover: novel tokens (not in any seed list), morphological edge cases,
    phonological interjection patterns, and suffix priority ordering.
    Returns True if all tests pass.
    """
    test_cases = [
        # Novel interjections (phonological rules, not vocabulary)
        ("ehm",       "INTJ"),
        ("ohhh",      "INTJ"),
        ("aaah",      "INTJ"),
        ("hmmmm",     "INTJ"),
        ("mmm",       "INTJ"),
        ("errrr",     "INTJ"),
        ("aduh",      "INTJ"),
        # Standard fillers
        ("uh",        "INTJ"),
        ("um",        "INTJ"),
        ("hmm",       "INTJ"),
        ("eh",        "INTJ"),
        ("anu",       "INTJ"),
        ("ee",        "INTJ"),
        ("eee",       "INTJ"),
        # Content words (should NOT be tagged as filler)
        ("rendang",   "CONTENT"),
        ("spaghetti", "CONTENT"),
        ("cappuccino","CONTENT"),
        ("goreng",    "CONTENT"),
        ("bakar",     "CONTENT"),
        # Verb morphology
        ("memesan",   "VERB"),
        ("dibatalkan","VERB"),
        ("mengganti", "VERB"),
        ("pesankan",  "VERB"),
        ("batalkan",  "VERB"),
        ("berikan",   "VERB"),
        # Nominal morphology (-nya priority over verb prefix)
        ("pesanannya","NOUN_VERB"),
        ("minumannya","NOUN_VERB"),
        ("makanannya","NOUN_VERB"),
        ("pesanan",   "NOUN_VERB"),
        # Pronouns
        ("saya",      "PRON"),
        ("gue",       "PRON"),
        ("kami",      "PRON"),
        # Address terms
        ("bang",      "ADDR"),
        ("mas",       "ADDR"),
        ("bro",       "ADDR"),
        ("sis",       "ADDR"),
        # Particles
        ("ya",        "PART"),
        ("deh",       "PART"),
        ("dong",      "PART"),
        ("nih",       "PART"),
        # Repair markers
        ("bukan",     "DM"),
        ("maaf",      "DM"),
        ("maksudnya", "DM"),
        ("wait",      "DM"),
        # Partial words
        ("na-",       "PARTIAL"),
        ("bi-",       "PARTIAL"),
        ("sp-",       "PARTIAL"),
        ("ay-",       "PARTIAL"),
        # Numbers
        ("satu",      "NUM"),
        ("dua",       "NUM"),
        ("tiga",      "NUM"),
    ]

    failures = []
    for token, expected in test_cases:
        got = infer_pos(token)
        if got != expected:
            failures.append((token, expected, got))

    print("\n" + "=" * 70)
    print(f"POS UNIT TESTS: {len(test_cases) - len(failures)}/{len(test_cases)} passed")
    print("=" * 70)

    if failures:
        print("\n  FAILURES:")
        for tok, expected, got in failures:
            print(f"    {tok:20s}  expected={expected:12s}  got={got}")
    else:
        print("  ✅  All tests passed.")

    return len(failures) == 0

In [10]:
output_path = "../data/vocabulary.json"

passed = run_pos_tests()
if not passed:
    print("Some POS unit tests failed. Review infer_pos() rules.")

sentences = [tokenize(r["text"]) for r in raw_records]
print("Loaded %d sentences, %d tokens total", len(sentences), sum(len(s) for s in sentences))

# ── Compute corpus stats (needed for report) ─────────────────────────────
corpus_stats = compute_corpus_stats(sentences, MIN_FREQ)

# ── Build vocabulary ─────────────────────────────────────────────────────
vocab = build_vocabulary(sentences, MIN_FREQ, PMI_THRESHOLD, ENTROPY_THRESHOLD)

# ── Write output ─────────────────────────────────────────────────────────
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)
print("Vocabulary written to %s", output_path)

# ── Reports ──────────────────────────────────────────────────────────────
print_pos_breakdown(sentences, MIN_FREQ)
print_report(vocab, corpus_stats)


POS UNIT TESTS: 51/51 passed
  ✅  All tests passed.
Loaded %d sentences, %d tokens total 1100 10514
Vocabulary written to %s ../data/vocabulary.json

POS TAG DISTRIBUTION (corpus tokens, freq >= min_freq)

  INTJ (9 types, 469 tokens):
    eh                    freq=168
    hmm                   freq=70
    eee                   freq=42
    anu                   freq=38
    ee                    freq=33
    mm                    freq=30
    uh                    freq=30
    hm                    freq=30
    um                    freq=28

  PART (8 types, 871 tokens):
    ya                    freq=571
    deh                   freq=84
    aja                   freq=72
    dong                  freq=42
    nih                   freq=37
    yaudah                freq=25
    sih                   freq=23
    gitu                  freq=17

  DM (9 types, 246 tokens):
    maaf                  freq=86
    bukan                 freq=44
    maksudnya             freq=29
    wait             